# Maia Lite 0.3.0 — A100 80 GB — pretraining preflight

**Objetivo:** Transformer decoder-only ~0,5B, inicialização do zero, contexto 2048, BF16. Este notebook realiza apenas **smoke test com dados sintéticos**. Não lê nem altera o Maia Corpus v8; não ignora `eligible_for_training=false`.

**Importante:** a aprovação de direitos dos dados reais ainda está pendente. O pré-treinamento de produção exige manifesto de dados elegíveis, revisão de tokenizer, orçamento de tokens e validação de checkpoints.

In [ ]:
import os, sys, json, math, time, random
from pathlib import Path
import torch
from torch import nn
import torch.nn.functional as F
print('Python:',sys.version.split()[0], 'PyTorch:',torch.__version__)
assert torch.cuda.is_available(), 'Selecione Runtime > Change runtime type > A100 GPU no Colab'
name=torch.cuda.get_device_name(0)
mem=torch.cuda.get_device_properties(0).total_memory/2**30
print('GPU:',name,'VRAM (GiB):',round(mem,1),'BF16:',torch.cuda.is_bf16_supported())
assert mem>=70, 'Este perfil foi dimensionado para A100 80GB'
assert torch.cuda.is_bf16_supported(), 'BF16 indisponível'
torch.manual_seed(20261104); random.seed(20261104)
torch.backends.cuda.matmul.allow_tf32=True


## Arquitetura proposta

36 blocos, largura 1024, 16 cabeças, FFN 4096, embeddings de entrada/saída compartilhados, posições aprendidas e atenção causal SDPA. **A contagem de parâmetros é calculada abaixo**, não presumida. Tokenizer GPT-2 (50.257) é apenas configuração provisória: **não assuma que é o tokenizer final do Corpus v8**.

In [ ]:
from dataclasses import dataclass
@dataclass
class Config:
    vocab_size:int=50257
    block_size:int=2048
    n_layer:int=36
    n_head:int=16
    n_embd:int=1024
    dropout:float=0.0
    bias:bool=True
cfg=Config()
class Block(nn.Module):
    def __init__(self,c):
        super().__init__()
        d=c.n_embd
        self.ln1=nn.LayerNorm(d)
        self.qkv=nn.Linear(d,3*d,bias=c.bias)
        self.proj=nn.Linear(d,d,bias=c.bias)
        self.ln2=nn.LayerNorm(d)
        self.fc=nn.Linear(d,4*d,bias=c.bias)
        self.out=nn.Linear(4*d,d,bias=c.bias)
        self.nh=c.n_head
    def forward(self,x):
        b,t,d=x.shape
        q,k,v=self.qkv(self.ln1(x)).chunk(3,dim=-1)
        q=q.view(b,t,self.nh,d//self.nh).transpose(1,2)
        k=k.view(b,t,self.nh,d//self.nh).transpose(1,2)
        v=v.view(b,t,self.nh,d//self.nh).transpose(1,2)
        a=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.0)
        x=x+self.proj(a.transpose(1,2).contiguous().view(b,t,d))
        x=x+self.out(F.gelu(self.fc(self.ln2(x)),approximate='tanh'))
        return x
class MaiaGPT(nn.Module):
    def __init__(self,c):
        super().__init__();self.cfg=c
        self.tok=nn.Embedding(c.vocab_size,c.n_embd)
        self.pos=nn.Embedding(c.block_size,c.n_embd)
        self.blocks=nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.norm=nn.LayerNorm(c.n_embd)
        self.head=nn.Linear(c.n_embd,c.vocab_size,bias=False)
        self.head.weight=self.tok.weight
        self.apply(self.init_weights)
    @staticmethod
    def init_weights(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,mean=0,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None:nn.init.zeros_(m.bias)
    def forward(self,idx,targets=None):
        b,t=idx.shape
        assert t<=self.cfg.block_size
        x=self.tok(idx)+self.pos(torch.arange(t,device=idx.device))
        for block in self.blocks:x=block(x)
        x=self.norm(x)
        logits=self.head(x)
        loss=None if targets is None else F.cross_entropy(logits.reshape(-1,logits.size(-1)).float(),targets.reshape(-1),ignore_index=-100)
        return logits,loss
model=MaiaGPT(cfg)
params=sum(p.numel() for p in model.parameters())
print('Parameters:',format(params,','),'(~%.3f B)'%(params/1e9))
assert 480_000_000<=params<=530_000_000
assert model.head.weight.data_ptr()==model.tok.weight.data_ptr()


## Smoke test BF16 e otimização

Começamos com sequência 256 e microbatch 1 para testar estabilidade. **Não são hiperparâmetros finais de produção.** O modelo ocupa memória substancial; em caso de OOM, reinicie a sessão antes de tentar novamente. AdamW mantém estados FP32; AMP BF16 não precisa de GradScaler.

In [ ]:
import gc
device='cuda'
model=model.to(device)
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4,betas=(0.9,0.95),weight_decay=0.1,fused=True)
seq_len=256
micro_batch=1
grad_accum=2
model.train(); optimizer.zero_grad(set_to_none=True)
t0=time.time()
for micro in range(grad_accum):
    tokens=torch.randint(0,cfg.vocab_size,(micro_batch,seq_len+1),device=device)
    with torch.autocast(device_type='cuda',dtype=torch.bfloat16):
        _,loss=model(tokens[:,:-1],tokens[:,1:])
    (loss/grad_accum).backward()
    print('microstep',micro+1,'loss',float(loss.detach()))
    del tokens,loss
norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
optimizer.step();optimizer.zero_grad(set_to_none=True)
torch.cuda.synchronize()
print('Step OK, grad_norm:',float(norm),'elapsed_s:',round(time.time()-t0,1))
print('Peak allocated GiB:',round(torch.cuda.max_memory_allocated()/2**30,2))


## Checkpoint recuperável (teste local)

Este exemplo grava pesos, optimizer, configuração e estados RNG no disco temporário do Colab. **Para produção, gravar em diretório persistente, com escrita atômica, hash e checkpoint de sampler/posição do dataset.** Não reinicie o Colab antes de baixar/copiar o checkpoint de teste se quiser preservá-lo.

In [ ]:
checkpoint=Path('/content/maia_lite_03_smoke_checkpoint.pt')
state={
 'model':model.state_dict(), 'optimizer':optimizer.state_dict(),
 'config':vars(cfg), 'global_step':1,
 'torch_rng':torch.get_rng_state(), 'cuda_rng':torch.cuda.get_rng_state_all(),
 'data_source':'SYNTHETIC_ONLY', 'rights_approval':False,
}
torch.save(state,checkpoint)
print('Saved:',checkpoint,'bytes:',checkpoint.stat().st_size)
assert torch.load(checkpoint,map_location='cpu',weights_only=True)['data_source']=='SYNTHETIC_ONLY'


## Gate obrigatório para dados reais

Não usar a simples presença do corpus ou do relatório de proveniência como aprovação. O treino de produção só pode ser conectado a manifestos de dados explicitamente aprovados e com inventário imutável. A célula abaixo demonstra o bloqueio, sem criar arquivos de autorização.

In [ ]:
from pathlib import Path
APPROVAL_PATH=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001/pretraining_production_v1_full/recovery_audit_v4/production_resume_v2/license_approval.json')
print('License approval present:',APPROVAL_PATH.is_file())
print('Production training is NOT enabled by this notebook.')
print('Before production: validate source-specific approvals, inventory SHA, tokenizer, data splits, and training manifest.')


## Plano de produção a definir

- Confirmar tokenizer final, tamanho do vocabulário, IDs especiais e taxa de tokens por idioma.
- Auditar **todas as fontes** do corpus, não apenas os 650 chunks restantes da Wikipedia PT.
- Selecionar apenas shards autorizados; validar integridade, deduplicação e split de validação.
- Medir throughput real com contexto 2048, checkpointing de gradientes, microbatch 1–4 e acumulação apropriada.
- Definir orçamento de tokens, LR schedule (warmup + cosine), checkpoints persistentes com RNG e cursor do dataloader, avaliação held-out e mecanismo de retomada.
- Submeter a política de publicação comercial dos pesos à revisão específica.
